# Dyniq-Qwen-1: QLoRA SFT of Qwen3-4B

**Step 1: environment check.** Before running, go to *Runtime → Change runtime type → T4 GPU* (or better).

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(props.total_memory / 1024**3, 2), "GB")
    print("Compute capability:", f"{props.major}.{props.minor}")
    # bf16 needs compute capability >= 8.0 (L4/A100 yes, T4 no -> use fp16)
    print("bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No GPU detected. Enable GPU in Runtime settings.")

PyTorch: 2.14.1+cpu
CUDA available: False
No GPU detected. Enable GPU in Runtime settings.


**Result (recorded):** Tesla T4, 14.56 GB, compute capability 7.5.
`is_bf16_supported()` says True, but on a T4 that is *emulated* bf16 (slow). T4 has no native bf16, so we use **fp16** everywhere.

## Step 2: install the training stack
Colab already ships PyTorch with CUDA, so we leave torch alone and add only what we need.

In [2]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets

import importlib
for pkg in ["torch", "transformers", "accelerate", "peft", "trl", "bitsandbytes", "datasets"]:
    print(f"{pkg:13s}", importlib.import_module(pkg).__version__)

torch         2.14.1+cpu
transformers  5.24.0
accelerate    1.16.0
peft          0.22.0
trl           1.18.2
bitsandbytes  0.50.2
datasets      5.2.0


## Step 3: fixed settings + seed
Everything that defines the experiment lives in one place so a rerun gives the same result.

In [3]:
import torch
from transformers import set_seed

EXPERIMENT   = "Dyniq-Qwen-1"
BASE_MODEL   = "Qwen/Qwen3-4B"
SEED         = 42
COMPUTE_DTYPE = torch.float16   # T4: no native bf16

set_seed(SEED)
print(EXPERIMENT, "|", BASE_MODEL, "| seed", SEED, "|", COMPUTE_DTYPE)

Dyniq-Qwen-1 | Qwen/Qwen3-4B | seed 42 | torch.float16


## Step 4: load Qwen3-4B in 4-bit (the "Q" in QLoRA)
- **NF4**: a 4-bit format designed for neural-net weights; the 4B model drops from ~8 GB (fp16) to ~2.5 GB.
- **Double quantization**: also compresses the scaling constants, saving a bit more memory.
- **Compute dtype fp16**: weights are stored in 4-bit but un-packed to fp16 for each matrix multiply.
- **sdpa attention**: PyTorch's built-in fast attention. FlashAttention 2 needs an Ampere+ GPU, so not on T4.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
    attn_implementation="sdpa",
)

print("Model memory:", round(model.get_memory_footprint() / 1024**3, 2), "GB")
print("GPU allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (3 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model memory: 2.42 GB
GPU allocated: 0.0 GB


## Step 5: smoke test the base model
Qwen3 expects its chat template. `enable_thinking=False` turns off its `<think>` reasoning block for a short, direct answer.
Keep this output: it's the "before" for our evaluation later.

In [5]:
messages = [{"role": "user", "content": "In two sentences, List all AI frontier labs in the United States?"}]

prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False)

print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

Fine-tuning a language model involves adjusting the model's parameters using a specific dataset to improve its performance on a particular task. This process allows the model to adapt to new data and better suit the needs of a specific application or domain.


**Baseline answer (recorded):** *"Fine-tuning a language model involves adjusting the model's parameters using a specific dataset to improve its performance on a particular task. This process allows the model to adapt to new data and better suit the needs of a specific application or domain."*

## Step 6: save to Google Drive
Colab sessions on the free tier disconnect. Checkpoints in Drive survive; files in `/content` don't.

In [6]:
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = f"/content/drive/MyDrive/dyniq/{EXPERIMENT}"
print("Saving to", OUTPUT_DIR)

Mounted at /content/drive
Saving to /content/drive/MyDrive/dyniq/Dyniq-Qwen-1


## Step 7: dataset engineering (general assistant)
We start from **UltraChat 200k**, a cleaned set of multi-turn conversations already in `messages` format (`role`/`content`), which is what Qwen's chat template and TRL both expect.

Choices for a T4:
- **Sample 3,000 conversations** (fixed seed): enough to shift style, small enough for about 1 hour of training.
- **Drop anything over 1,024 tokens**: long examples are what cause out-of-memory errors on a 15 GB card.
- **Hold out 100** for an evaluation loss, so we can spot overfitting.

Later we will mix in our own curated reasoning, coding and maths examples in the same format.

In [7]:
from datasets import load_dataset

MAX_LEN   = 1024
N_SAMPLES = 3000

raw = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
raw = raw.shuffle(seed=SEED).select(range(N_SAMPLES * 2))   # oversample, then filter

def n_tokens(example):
    ids = tokenizer.apply_chat_template(example["messages"], tokenize=True, enable_thinking=False)
    return {"n_tokens": len(ids)}

raw = raw.map(n_tokens)
filtered = raw.filter(lambda x: x["n_tokens"] <= MAX_LEN).select(range(N_SAMPLES))
filtered = filtered.remove_columns([c for c in filtered.column_names if c != "messages"])

split = filtered.train_test_split(test_size=100, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print(train_ds, eval_ds, sep="\n")
print("\nExample, as the model will see it:\n")
print(tokenizer.apply_chat_template(train_ds[0]["messages"], tokenize=False, enable_thinking=False)[:1200])

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…): reconstructing file:   0%|          |  0.00B /  244MB            

data/train_sft-00000-of-00003-a3ecf92756(…): downloading bytes:           |  0.00B            

data/train_sft-00001-of-00003-0a1804bcb6(…): reconstructing file:   0%|          |  0.00B /  244MB            

data/train_sft-00001-of-00003-0a1804bcb6(…): downloading bytes:           |  0.00B            

data/train_sft-00002-of-00003-ee46ed25cf(…): reconstructing file:   0%|          |  0.00B /  244MB            

... (14 outputs left out)

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6000 [00:00<?, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 2900
})
Dataset({
    features: ['messages'],
    num_rows: 100
})

Example, as the model will see it:

<|im_start|>user
Produce a 30-minute podcast episode focused on providing practical tips and strategies for remote workers to maintain motivation and productivity in their home office environment. In your episode, explore the psychology of motivation, discuss the common challenges of working from home and offer listeners some actionable steps they can take to stay on task, stay connected and stay inspired. Consider featuring expert guests, case studies or personal stories to illustrate your points and make your content engaging and relatable to a broad audience. Finally, be sure to end on a positive and inspiring note that encourages listeners to take ownership of their work-life balance and thrive in their remote work setup.<|im_end|>
<|im_start|>assistant
(Introduction music)

Host: Hello and welcome to this week’s episode of The 

## Step 8: LoRA configuration (the "LoRA" in QLoRA)
The 4-bit base weights stay **frozen**. We train small low-rank matrices added to every attention and MLP projection.
- **r = 16**: adapter rank, i.e. its capacity. 16 is a good default for style/behaviour tuning.
- **lora_alpha = 32**: scales the adapter's effect (the usual rule of thumb is 2 x r).
- **dropout 0.05**: light regularisation.
- **target_modules**: all 7 linear layers in each Qwen3 block. Covering all of them works better than attention-only.

In [8]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

## Step 9: training configuration (T4-sized)
- **Batch 2 x gradient accumulation 8 = effective batch 16**: what fits in memory vs. what the optimiser sees.
- **Gradient checkpointing**: recomputes activations instead of storing them. About 30% slower, but uses much less memory.
- **fp16 = True, bf16 = False**: the T4 constraint from Step 1.
- **paged_adamw_8bit**: an 8-bit optimiser that can spill to CPU RAM instead of crashing.
- **lr 2e-4, cosine schedule, 1 epoch**: standard QLoRA settings. More epochs on 3k examples tends to overfit.

In [9]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    max_length=MAX_LEN,
    packing=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)
# fp16 mixed precision needs the *trainable* (LoRA) weights in fp32.
# Some came out as bf16, which crashes the fp16 gradient scaler, so cast them.
from collections import Counter
for name, param in trainer.model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()

print("Trainable dtypes:", Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad))
trainer.model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


Tokenizing train dataset:   0%|          | 0/2900 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2900 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2900 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2900 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Trainable dtypes: Counter({'torch.float32': 504})
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


## Step 10: fine-tune
Watch two numbers: **training loss** should fall steadily. **eval loss** should fall too; if it starts rising while training loss keeps falling, the model is overfitting.
If the session disconnects, rerun Steps 2-9, then this cell. It resumes from the last checkpoint in Drive automatically.

In [ ]:
import glob

required = {"EXPERIMENT": "Step 3", "model": "Step 4", "OUTPUT_DIR": "Step 6",
            "train_ds": "Step 7", "lora_config": "Step 8", "trainer": "Step 9"}
missing = [f"{step} ({var})" for var, step in required.items() if var not in globals()]
if missing:
    raise RuntimeError("Run these cells first, in order: " + ", ".join(missing))

resume = bool(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
print("Resuming from checkpoint" if resume else "Starting fresh")
trainer.train(resume_from_checkpoint=resume)

Starting fresh


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


## Step 11: save the adapter + quick before/after check
The adapter is only about 130 MB. It's the only new thing we've made; the base model is downloaded again whenever it's needed.

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to", ADAPTER_DIR)

trainer.model.eval()
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
with torch.no_grad():
    out = trainer.model.generate(**inputs, max_new_tokens=120, do_sample=False)
print("\nFine-tuned answer:\n")
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Step 12: serve Dyniq as an API (testing via a tunnel)
**First do Runtime > Restart session**, so the training copy of the model is no longer using GPU memory. Then run only the cells below.

Before running:
1. Upload `app.py` from your project folder with the Files panel on the left, into `/content`.
2. Add your API key to Colab's **Secrets** panel (key icon, left sidebar) as `DYNIQ_API_KEY`. Copy the value from your local `.env` file and enable notebook access. Never paste the key into a cell.

The tunnel URL changes every session and dies when Colab stops, so this setup is for testing only.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes flask

import os, re, subprocess, time
import requests
from google.colab import drive, userdata
drive.mount("/content/drive")

env = dict(os.environ,
           DYNIQ_API_KEY=userdata.get("DYNIQ_API_KEY"),
           BASE_MODEL="Qwen/Qwen3-4B",
           ADAPTER_PATH="/content/drive/MyDrive/dyniq/Dyniq-Qwen-1/adapter",
           PORT="5001")
server = subprocess.Popen(["python", "/content/app.py"], env=env,
                          stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)

for _ in range(120):                      # wait up to ~10 min for the model to load
    try:
        print(requests.get("http://localhost:5001/health", timeout=2).json())
        break
    except Exception:
        time.sleep(5)
else:
    print(open("/content/server.log").read()[-3000:])

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:5001"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0)
        break
print("Public API URL:", url)

In [ ]:
r = requests.post(f"{url}/chat",
                  headers={"x-api-key": userdata.get("DYNIQ_API_KEY")},
                  json={"messages": [{"role": "user", "content": "Hi! Who are you?"}]},
                  timeout=120)
print(r.status_code, r.json())